# Predict future service demand

Apply the saved demand model to the complete 200 m and 500 m scenario-feature tables. Use the separate hurdle Tweedie R model to examine demand relationships and fit diagnostics. Operational simulation uses the demand forecasts. Retraining is unnecessary when reproducing the packaged case.

Select a Python kernel with the required packages; use the dependency profile in requirements.txt for the saved demand model. The reference installation uses the **ArcPy** kernel (Anaconda base). Run the cells in order. Shared settings are in `desktop_settings.json`; the default result folder is `../outputs/desktop_run_en`. Run the first setup cell after reopening an updated notebook so the helper code is refreshed. The scripts check dependencies without installing packages. Starting validation or a complete run preserves incompatible previous results in their original folder and selects a fresh run ID when required.

In [ ]:
from pathlib import Path
import sys
# Open notebooks within the toolkit; editor scripts can also locate their own folder.
_entry_dir = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
ROOT = next((p for p in [_entry_dir, *_entry_dir.parents, Path.cwd(), *Path.cwd().parents]
    if (p/'desktop_scripts/desktop_helpers.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from inside the toolkit folder, then rerun this cell.')
sys.path.insert(0, str(ROOT/'desktop_scripts'))
import importlib
import desktop_helpers
importlib.reload(desktop_helpers)
from desktop_helpers import check_environment, run, run_r, get_output_dir, get_study_config
import pandas as pd
versions = check_environment()
OUT = get_output_dir()

Run the demand-estimation R script if it has not yet been completed for this run ID. Skip this cell when the model has already been fitted in RStudio.

In [ ]:
OUT = run_r('03_Demand_Estimation.R')
print((OUT/'demand_summary.txt').read_text(encoding='utf-8',errors='replace'))

Regenerate both complete demand forecasts using the saved model.

In [ ]:
OUT = run('predict')
print(pd.read_csv(OUT/'ml_prediction_summary.csv').to_string(index=False))

Inspect hourly forecasts and a sample of predicted OD demand.

In [ ]:
case = str(get_study_config()['demand']['prediction_cases'][0])
print(pd.read_csv(OUT/f'ml_hourly_{case}.csv').to_string(index=False))
print(pd.read_csv(OUT/f'ml_demand_{case}.csv.gz',nrows=10).to_string(index=False))